**В этом задании вам предстоит собрать данные с сайта lifehacker.ru из рубрики Технологии с помощью библиотек requests и BeautifulSoup.
В частности, нужно собрать заголовки и тексты материалов с первых десяти страниц рубрики.**

In [ ]:
# Импорт модулей
import requests
from bs4 import BeautifulSoup
import pandas as pd
from google.colab import drive
drive.mount('drive')

# Список для хранения данных
data = []

In [ ]:


# Проходим по страницам с 1 по 10
for page in range(1, 11):
    url = f'https://lifehacker.ru/topics/technology/?page={page}'
    response = requests.get(url)

    # Проверка успешности запроса
    if response.status_code == 200:

        # Прочитать код веб-страницы
        soup = BeautifulSoup(response.text, 'html.parser')

        # На полученных страницах найти ссылки на статьи
        article_links = soup.find_all('a', class_='lh-small-article-card__link')

        # Нужно убрать рекламные страницы из списка ссылок
        article_links = list(filter(lambda link: not '/special/' in link['href'], article_links))

        # Цикл по сслыкам
        for link in article_links:

            article_url = 'https://lifehacker.ru' + link['href']
            article_response = requests.get(article_url)

            # Проверяется есть ли какой-то ответ
            if article_response.status_code == 200:
                # print(f"Статья загружена: {article_url}")

                # Прочитать код веб-страницы
                article_soup = BeautifulSoup(article_response.text, 'html.parser')

                # Получить заголовок
                title = article_soup.find('h1', class_='article-card__title')
                title_text = title.text.strip() if title else 'Без заголовка'

                # Прочитать текст статьи
                article_content = article_soup.find('article')
                content_text = article_content.get_text(separator='\n').strip() if article_content else 'Без текста'

                # Добавляем данные в список
                data.append([page, title_text, content_text])
            else:
                print(f"Не удалось загрузить статью: {article_url}, статус код: {article_response.status_code}")
    else:
        print(f"Не удалось загрузить страницу: {url}, статус код: {response.status_code}")

# Создаем DataFrame
df = pd.DataFrame(data, columns=['Номер страницы', 'Заголовок', 'Текст статьи'])

In [ ]:
# Посмотреть
df[:3]

,Номер страницы,Заголовок,Текст статьи
0,1,14 удобных инструментов для удалённой работы в...,1. Рабочий мессенджер: Telegram\nПлатформы:\n ...
1,1,Google будет скрывать почту пользователей Andr...,Google \nработает\n над функцией под названием...
2,1,Casio представила часы-кольцо CRW-001-1JR с се...,Casio \nпредставила\n крошечные часы в форме к...


In [ ]:
# Сохранить
df.to_csv('lifehacker.csv', index=False, encoding='utf-8-sig')
!cp lifehacker.csv "drive/My Drive/"